# Simple Colab GPU Indexer

## What this does:
1. **Uploads** your phonex project
2. **Installs** packages
3. **Runs** the indexer on GPU (fast!)
4. **Downloads** the index

---

## IMPORTANT: Before you start

**Set GPU in Colab:**
- Go to `Runtime` menu → `Change runtime type` → Select `GPU` → Save

---

## Step 1: Check GPU
This verifies GPU is enabled.

In [1]:
import torch

print("GPU Check:")
print(f"GPU Available? {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("\n✅ GPU is READY")
else:
    print("\n❌ NO GPU! Go to Runtime > Change runtime type > GPU")

GPU Check:
GPU Available? True
GPU Name: Tesla T4
GPU Memory: 15.6 GB

✅ GPU is READY


## Step 2: Upload your phonex.zip

**Before running this cell:**
1. On your computer, go to `D:\Gitrnd\phonex\`
2. Select the entire `phonex` folder
3. Right-click → Send to → Compressed (zipped) folder
4. This creates `phonex.zip`

Then run the cell below and choose `phonex.zip` to upload.

In [ ]:
print("MANUAL UPLOAD INSTRUCTIONS:")
print("=" * 70)
print()
print("1. Look at the LEFT SIDEBAR of Colab")
print("2. Click the FOLDER ICON (📁) at the top")
print("3. This opens the FILE EXPLORER")
print()
print("4. You should see:")
print("   /content/")
print()
print("5. Right-click in the empty space and select 'Upload'")
print("6. Choose your phonex.zip file from your computer")
print()
print("7. Wait for it to upload (shows progress bar)")
print()
print("=" * 70)
print()
print("Once uploaded, run the next cell to extract it:")


UPLOAD INSTRUCTIONS:
1. Click the PLAY button (▶️) to the left of this cell
2. A 'Choose Files' button will appear below
3. Click it and select your phonex.zip
4. Wait for upload to complete

Waiting for file upload...



KeyboardInterrupt: 

In [15]:
import os
print("ZIP files in /content/:")
for f in os.listdir('/content/'):
    if f.endswith('.zip'):
        print(f"  ✅ {f}")
        size = os.path.getsize(f'/content/{f}') / (1024*1024)
        print(f"     Size: {size:.1f} MB")

ZIP files in /content/:


In [20]:
import zipfile
import os
import subprocess

print("Searching for ZIP files...")
print()

# Use command line to find all ZIP files
result = subprocess.run(['find', '/content/', '-name', '*.zip', '-type', 'f'], 
                       capture_output=True, text=True)
zip_files = result.stdout.strip().split('\n')
zip_files = [f for f in zip_files if f]  # Remove empty strings

if zip_files:
    zip_path = zip_files[0]
    print(f"✅ Found ZIP: {zip_path}")
    print(f"Extracting...")
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    
    print("✅ Extraction done!")
    os.remove(zip_path)
    print("✅ Removed ZIP file")
    
    print("\n📁 Folders now in /content/:")
    for item in sorted(os.listdir('/content/')):
        path = f'/content/{item}'
        if os.path.isdir(path):
            print(f"   📁 {item}/")
else:
    print("No ZIP files found.")
    print("\n📁 Everything in /content/:")
    subprocess.run(['find', '/content/', '-maxdepth', '3'])
    
    print("\n" + "="*60)
    print("If you can see phonex.zip in the file explorer,")
    print("please share the output above so we can find it.")

Searching for ZIP files...

No ZIP files found.

📁 Everything in /content/:

If you can see phonex.zip in the file explorer,
please share the output above so we can find it.


## Step 3: Install packages
This installs everything needed for indexing (llama-index, chromadb, embeddings).

print("╔" + "="*60 + "╗")
print("║ EASY FIX: Use Google Drive                                  ║")
print("╚" + "="*60 + "╝")
print()
print("Step 1: Upload phonex.zip to your Google Drive")
print("        - Open drive.google.com")
print("        - Upload phonex.zip there")
print()
print("Step 2: Mount Google Drive (run the code below):")
print()
print("   from google.colab import drive")
print("   drive.mount('/content/drive')")
print()
print("Step 3: Copy the ZIP to Colab:")
print()
print("   import shutil")
print("   shutil.copy('/content/drive/My Drive/phonex.zip', '/content/phonex.zip')")
print()
print("Step 4: Then run cell 4 (extraction)")
print()
print("="*62)

In [21]:
print("Step 1: Mount Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted at /content/drive/")
print("\nNow check Google Drive for phonex.zip:")
import os
drive_files = os.listdir('/content/drive/My Drive/')
zips = [f for f in drive_files if f.endswith('.zip')]

if zips:
    print(f"\n✅ Found: {zips}")
    print("\nStep 2: Copying to Colab...")
    import shutil
    for z in zips:
        src = f'/content/drive/My Drive/{z}'
        dst = f'/content/{z}'
        shutil.copy(src, dst)
        print(f"   Copied {z}")
    print("\n✅ Done! Now run cell 4 (extraction)")
else:
    print("\n❌ No ZIP files in Google Drive")
    print("Please upload phonex.zip to Google Drive first")

Mounted at /content/drive

✅ Google Drive mounted at /content/drive/

Now check Google Drive for phonex.zip:

✅ Found: ['infra-azure-vplx-master (1).zip', 'Vanguard Software Architecture and Design (SAD)_new.doc.zip', 'phonex.zip']

Step 2: Copying to Colab...
   Copied infra-azure-vplx-master (1).zip
   Copied Vanguard Software Architecture and Design (SAD)_new.doc.zip
   Copied phonex.zip

✅ Done! Now run cell 4 (extraction)


In [23]:
print("Installing packages...")
!pip install -q llama-index chromadb sentence-transformers llama-index-vector-stores-chroma jedi
print("✅ All packages installed successfully!")
print()
print("Ready to proceed to the next step.")

Installing packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.0 MB/s eta 0:00:00
✅ All packages installed successfully!

Ready to proceed to the next step.


## Step 4: Run the indexer

This runs `index_codebase.py`.

**What happens:**
- Finds your source code
- Splits it into chunks
- Creates embeddings (using GPU)
- Saves Chroma index

**This takes time** (could be 15 minutes to hours depending on code size). Be patient!

In [24]:
import os
import subprocess

# Find and run the indexer
indexer = None

if os.path.exists('/content/phonex/index_codebase.py'):
    indexer = '/content/phonex/index_codebase.py'
elif os.path.exists('/content/index_codebase.py'):
    indexer = '/content/index_codebase.py'

if indexer:
    print(f"Found indexer at: {indexer}")
    print("\nRunning...\n")
    print("=" * 70)
    subprocess.run(['python', indexer])
    print("=" * 70)
    print("\n✅ Indexing complete!")
else:
    print("ERROR: Could not find index_codebase.py")
    print("Make sure phonex.zip was uploaded and extracted.")

ERROR: Could not find index_codebase.py
Make sure phonex.zip was uploaded and extracted.


In [ ]:
print("📁 Contents of /content/:")
import os
for item in sorted(os.listdir('/content/')):
    path = f'/content/{item}'
    if os.path.isdir(path):
        print(f"   📁 {item}/")
        # Show what's inside each folder
        try:
            contents = os.listdir(path)
            for sub in contents[:5]:  # Show first 5 items
                print(f"      - {sub}")
            if len(contents) > 5:
                print(f"      ... and {len(contents)-5} more")
        except:
            pass
    else:
        print(f"   📄 {item}")

print("\n" + "="*60)
print("If you don't see 'phonex' folder above,")
print("run the extraction cell (cell 4) first!")

## Step 5: Download the index

This zips the vector database and downloads it to your computer.

In [ ]:
import os
import shutil
from google.colab import files

# Find the index
index_dir = '/tmp/phonex_index'

if os.path.exists(index_dir):
    print(f"Index location: {index_dir}")
    
    # Count files and size
    total_size = sum(os.path.getsize(os.path.join(dirpath, filename)) for dirpath, dirnames, filenames in os.walk(index_dir) for filename in filenames)
    print(f"Index size: {total_size / 1024**2:.1f} MB")
    
    # Zip it
    print("\nZipping...")
    shutil.make_archive('/tmp/phonex_index_download', 'zip', index_dir)
    
    # Download
    print("Downloading...")
    files.download('/tmp/phonex_index_download.zip')
    print("\n✅ Download complete!")
    print("\nYour vector database is ready to use locally.")
else:
    print(f"ERROR: Index not found at {index_dir}")
    print("Check that indexing completed successfully.")

## Done!

### What you got:
- **phonex_index_download.zip** - Your vector database (Chroma index)

### Next steps:
1. Extract the ZIP on your local machine
2. Update your local `index_codebase.py` to point to the extracted folder
3. Run queries locally (will be fast - no GPU needed for queries)